# SSD MobileNet V2 (512×512) — PhenoBench QAT (qat)

Quantization-aware training (**full int8 (backbone + head), folded** — per-tensor weights) for
**multi-class crop-vs-weed instance detection**, resuming from the companion *finetune* notebook's
`finetune/` export and publishing a self-contained `qat/` artifact.

This is one of two sibling QAT notebooks (`qat_per-tensor`, `qat_per-channel`) for this
config; they share the scheme and differ only in weight granularity, so they are
independent and individually re-runnable.

## Inputs

- **Finetune output** — attach this config's finetune notebook output via
  *Add Input → Your Work / Notebook Output*. Kaggle mounts it at
  `/kaggle/input/notebooks/freimutdiener/ssd-mn2-mc-phenobench-512-finetune`; its `finetune/` (zoo layout: `checkpoint/` +
  `pipeline.config`) is the QAT `model_path`.
- The PhenoBench dataset bundle (`train.record` / `val.record` / `label_map`).

## Resolution ladder

This is the **512×512** rung of the input-resolution study; the reference
rung is `21_ssd-mn2_mc_phenobench_320_qat_per-tensor`.

**Nothing about the quantization regime changes across the ladder.** Resolution,
batch size and step budget are *inherited* from the attached finetune's manifest
(`stages.finetune.config`) — this notebook sets none of them, exactly as it
already inherits anchor scales, matcher thresholds and NMS. The only thing that
makes this the 512 rung is **which finetune output is attached as an input**.

Section 4 asserts the inherited resolution rather than trusting it: attaching
the wrong rung's output is one wrong click in the Kaggle input picker, and
nothing else in the notebook would contradict it.

## Output contract (`/kaggle/working`)

On completion the working directory holds exactly:

```
/kaggle/working/
├── manifest.qat.json    # fragment manifest: only the `qat` stage
└── qat_per-tensor/
    ├── checkpoint/          # fake-quantized model-only ckpt-0
    ├── saved_model/         # SavedModel (fp32-simulated quant graph)
    ├── graphs/              # training / validation curves (PDF)
    ├── pipeline.config      # as-run TFOD pipeline
    ├── best_metric.json
    └── metrics_history.json
```

The fragment is merged into the finetune's `manifest.json` after the fact
(`ExperimentManifest.merge`), keeping each `ptq/qat_per-{tensor,channel}` run decoupled.

## 1 · Environment

Target runtime (Kaggle, "Pin to original environment" enabled):

- Python 3.10 · TensorFlow 2.11 · CUDA-enabled GPU

In [ ]:
!python -V

In [ ]:
!pip install --no-cache-dir --no-deps \
  tf_slim \
  pycocotools \
  lvis \
  contextlib2 \
  gin-config \
  tf-models-official==2.13.2 \
  tensorflow-model-optimization==0.7.5 \
  git+https://github.com/frdiener/agri-vision-edge.git

In [ ]:
import json
import shutil
from pathlib import Path
from dataclasses import asdict
import matplotlib.pyplot as plt

# setup_tensorflow_models() must run before anything imports object_detection.
from agri_vision_edge.third_party import setup_tensorflow_models
setup_tensorflow_models()

from agri_vision_edge.experiment import (
    ExperimentManifest,
    capture_environment,
)
from agri_vision_edge.experiment import (
    AugmentationConfig,
    FineTuneConfig,
)

# The shared trainer: one config object drives finetune / QAT + export.
from agri_vision_edge.tfod_trainer import (
    FinetuneRunConfig,
    TrainingControlConfig,
    run_finetune,
    write_pipeline,
    export_run,
)

# Curves are read back from the trainer's metrics_history.json (no TensorBoard).
from agri_vision_edge.evaluation.curves import (
    load_history_scalars,
    available_tags,
    plot_metric_curves,
    plot_loss_curves,
    plot_learning_rate,
    plot_steps_per_second,
    plot_map_curves,
    plot_recall_curves,
)

## 2 · Output paths

`tfod_trainer` writes its working tree into a `/tmp` scratch dir; the published
`/kaggle/working` holds only the `manifest.qat.json` fragment and the
assembled `qat/` artifact.

In [ ]:
WORK = Path("/kaggle/working")

# Per-channel weights: False = per-tensor (i.MX8M Plus Vivante/Teflon NPU);
# True = per-channel (i.MX93 Arm Ethos-U65, which accepts it). This is the ONLY
# switch to flip -- it appends a "_per-channel" identifier to the output folder /
# manifest stage / fragment / artifacts, so a per-channel run never clobbers the
# per-tensor one. Convert with the matching Per-Channel switch in
# tflite_conversion.py.
QAT_PER_CHANNEL = False

LABEL = "qat" + ("_per-channel" if QAT_PER_CHANNEL else "_per-tensor")

QAT_DIR = WORK / LABEL
GRAPHS_PATH = QAT_DIR / "graphs"
MANIFEST_PATH = WORK / f"manifest.{LABEL}.json"

# Finetune export consumed as the QAT starting point (model_path). The finetune
# notebook publishes its zoo-layout export (checkpoint/ + pipeline.config) under
# `finetune/`.
FINETUNE_DIR = Path("/kaggle/input/notebooks/freimutdiener/ssd-mn2-mc-phenobench-512-finetune")
FINETUNE_EXPORT = FINETUNE_DIR / "finetune"

# Intermediate scratch (kept off /kaggle/working so it is never published).
RUN_DIR = Path(f"/tmp/ave_{LABEL}_run")

GRAPHS_PATH.mkdir(parents=True, exist_ok=True)
assert FINETUNE_EXPORT.exists(), (
    f"Finetune export not found at {FINETUNE_EXPORT} -- attach this config's "
    "finetune notebook output as an input."
)

## 3 · Experiment manifest (fragment)

A **fragment** manifest carrying only this QAT stage. It is merged into the
finetune's full `manifest.json` later (`ExperimentManifest.merge`, which merges
disjoint stages + artifacts and raises on a duplicate). The QAT run's
environment is recorded inside the stage `runtime` so it survives the merge.

In [ ]:
manifest = ExperimentManifest(
    name="ssd-mobilenet-v2_mc_phenobench_512x512",
    task="object_detection",
)

env = capture_environment()
manifest.set_environment(platform="kaggle", **env)

## 4 · QAT configuration

The pipeline tuning (anchors, matcher thresholds, base augmentation, NMS) is
**inherited from the finetune's manifest** (`stages.finetune.config`) so the
anchor semantics the head was trained against are preserved exactly. Only the
QAT-specific knobs are overridden:

- **Optimization** softened — QAT resumes from converged fp32 weights, so a low
  learning rate and a short schedule suffice for the weights to adapt to the
  fake-quantization nodes.
- **Augmentation** softened — photometric jitter and random crop/zoom dropped,
  keeping only the geometric (flip / 90° rotation) symmetries.
- **Scheme:** full int8, backbone + detection head — full int8 (backbone + head), folded

In [ ]:
from dataclasses import fields

# Inherit the finetune's pipeline tuning (anchors, matcher thresholds, base
# augmentation, NMS) from its manifest, so the anchor semantics the head was
# trained against are preserved exactly. Only the QAT knobs are overridden.
stage_config = (
    ExperimentManifest.load(FINETUNE_DIR / "manifest.json")
    .data["stages"]["finetune"]["config"]
)
# Accept both the whole-run mapping (new: a FinetuneRunConfig.to_mapping() with a
# nested "finetune") and the legacy flat FineTuneConfig dict.
finetune_config = dict(stage_config.get("finetune", stage_config))

# Soften augmentation for QAT: drop photometric jitter and random crop/zoom,
# keep the geometric (flip / 90-degree rotation) symmetries.
augmentation = AugmentationConfig(**finetune_config.pop("augmentation"))
augmentation.random_crop = False
augmentation.zoom_range = None
augmentation.brightness_max_delta = None
augmentation.contrast_range = None
augmentation.saturation_range = None
augmentation.jpeg_quality_range = None

# Keep only FineTuneConfig fields (drops any legacy early_stopping / control keys
# that older manifests stored alongside the pipeline tuning).
_ft_fields = {f.name for f in fields(FineTuneConfig)} - {"augmentation"}
finetune_config = {k: v for k, v in finetune_config.items() if k in _ft_fields}

config = FineTuneConfig(**finetune_config, augmentation=augmentation)

# Everything that defines this rung -- resolution, batch size, step
# budget -- came from the finetune manifest just above; nothing in this
# notebook sets them. That makes the attached input the only thing
# distinguishing this from the 320 notebook, so verify it: a wrong
# attachment would train at the wrong resolution under a 512 name and
# only surface much later as an inexplicable benchmark result.
EXPECTED_IMAGE_SIZE = 512
assert config.image_size == EXPECTED_IMAGE_SIZE, (
    f"attached finetune is {config.image_size}px, but this is the "
    f"{EXPECTED_IMAGE_SIZE}px rung -- check the notebook input at {FINETUNE_DIR}"
)

print(
    f"inherited: image_size={config.image_size} "
    f"batch_size={config.batch_size} num_steps={config.num_steps}"
)

# The stage's full config (the whole FinetuneRunConfig mapping) is committed
# below, once the run config is assembled.
manifest.add_stage(LABEL, runtime={"environment": env})

"defined"

## 5 · Dataset

A `tfod_trainer` *dataset bundle* is a directory holding `label_map.pbtxt`,
`train.record` and `val.record` — exactly the layout of the mounted
`mc-phenobench` dataset. The raw PhenoBench images are used only for
qualitative evaluation at the end.

In [ ]:
manifest.set_dataset(
    name="phenobench",
    train_split="train",
    validation_split="val",
    num_classes=2,
)

dataset_dir = Path("/kaggle/input/datasets/freimutdiener/mc-phenobench-no-partials")
dataset_raw_dir = Path("/kaggle/input/datasets/freimutdiener/phenobench-raw-dataset-v1-1-0/PhenoBench")

label_map_path = dataset_dir / "label_map.pbtxt"
test_imgs = list((dataset_raw_dir / "test" / "images").glob("*.png"))

print(f"{len(test_imgs)} test images loaded")

## 6 · Starting point and run configuration

The QAT `model_path` is the finetune's `finetune/` export — a drop-in zoo-layout
base model (`checkpoint/ckpt-0.*` + `pipeline.config`). `run_finetune` folds the
BatchNorms and inserts the fake-quant nodes (`qat=True`, backbone + head) on top
of the restored weights.

In [ ]:
manifest.set_checkpoint(
    # Deliberately unchanged across the ladder: the TF2 zoo publishes no
    # plain SSD-MobileNetV2 checkpoint above 320x320, so every rung
    # ultimately descends from these same COCO weights. This records that
    # origin, not this notebook's input resolution.
    model_name="ssd_mobilenet_v2_320x320_coco17_tpu-8",
    pretrained_dataset="phenobench (finetune)",
    source="finetune_ptq_export",
)

# The finetune export, in TF model-zoo layout, is the QAT base model.
MODEL_DIR = FINETUNE_EXPORT

In [ ]:
from agri_vision_edge.tfod_trainer import TrainingControlConfig

run_config = FinetuneRunConfig(
    model_path=MODEL_DIR,
    dataset_bundle_path=dataset_dir,
    num_classes=2,
    output_dir=RUN_DIR,      # scratch; the published artifact is assembled into QAT_DIR
    finetune=config,
    control=TrainingControlConfig(
        #
        # Quantization-aware training: the full int8 scheme (fold BN + fake-quant
        # backbone + head).
        #
        qat=True,
        # Per-channel vs per-tensor weights -- set via QAT_PER_CHANNEL above (it
        # also drives the output folder / manifest stage identifier).
        qat_per_channel=QAT_PER_CHANNEL,
        # Treat PhenoBench partial (border / low-visibility) plants as
        # do-not-care during the continuous eval (matches `ave evaluate
        # --ignore-partials`). No-op on bundles built without partial flags.
        eval_ignore_partials=True,

        # QAT optimization schedule -- low LR, short run from converged fp32 weights.
        warmup_epochs=1,
        lr_plateau=True,
        lr_plateau_base_lr=2e-4,
        lr_plateau_warmup_lr=1e-4,
        lr_plateau_factor=0.5,
        lr_plateau_patience=6,
        lr_plateau_cooldown=1,
        lr_plateau_min_lr=5e-5,
        lr_plateau_min_delta=1e-3,
        lr_plateau_restore_best=True,
        lr_plateau_exhausted_patience=1,
    ),
)

# Commit the whole run config (orchestration + finetune + control) to the stage,
# so it round-trips through FinetuneRunConfig.from_mapping.
manifest.update_stage(LABEL, config=run_config.to_mapping())

## 7 · Training

`run_finetune` builds the detection model, restores the COCO checkpoint, and
runs the training loop with metric-based checkpointing and early stopping,
appending one record per logged step to `metrics_history.json` and saving the
best checkpoint (by validation mAP) under the scratch run directory.

In [ ]:
result = run_finetune(run_config)

In [ ]:
# Promote the metric files from the scratch run dir into the qat artifact.
shutil.copy2(result.best_metric_path, QAT_DIR / "best_metric.json")
shutil.copy2(result.history_path, QAT_DIR / "metrics_history.json")

best = json.loads((QAT_DIR / "best_metric.json").read_text())
print(
    f"Best {best['metric_name']}: {best['metric_value']:.5f} "
    f"at step {best['step']}"
)

manifest.update_stage(LABEL, metrics={"best_metric": best})
manifest.update_stage(
    LABEL,
    artifacts={
        "best_metric_json": f"{LABEL}/best_metric.json",
        "metrics_history_json": f"{LABEL}/metrics_history.json",
    },
)

## 8 · Training and validation curves

QAT curves read back from `qat/metrics_history.json`; each figure is saved
into `qat/graphs/` as a vector PDF.

In [ ]:
history_df = load_history_scalars(QAT_DIR / "metrics_history.json")

print("Available metric tags:")
for tag in available_tags(history_df):
    print("-", tag)

In [ ]:
loss_fig, _ = plot_loss_curves(
    history_df,
    save_path=GRAPHS_PATH / "training_loss_curves.pdf",
)
display(loss_fig)

In [ ]:
lr_fig, _ = plot_learning_rate(
    history_df,
    save_path=GRAPHS_PATH / "learning_rate_schedule.pdf",
)
display(lr_fig)

In [ ]:
tput_fig, _ = plot_steps_per_second(
    history_df,
    save_path=GRAPHS_PATH / "training_throughput.pdf",
)
display(tput_fig)

In [ ]:
map_fig, _ = plot_map_curves(
    history_df,
    save_path=GRAPHS_PATH / "eval_precision_curves.pdf",
)
display(map_fig)

In [ ]:
recall_fig, _ = plot_recall_curves(
    history_df,
    save_path=GRAPHS_PATH / "eval_recall_curves.pdf",
)
display(recall_fig)

In [ ]:
size_precision_fig, _ = plot_metric_curves(
    df=history_df,
    tags=[
        "DetectionBoxes_Precision/mAP (small)",
        "DetectionBoxes_Precision/mAP (medium)",
        "DetectionBoxes_Precision/mAP (large)",
    ],
    title="Validation Precision per Object Size",
    ylabel="mAP",
    save_path=GRAPHS_PATH / "eval_precision_size_curves.pdf",
)
display(size_precision_fig)

In [ ]:
size_recall_fig, _ = plot_metric_curves(
    df=history_df,
    tags=[
        "DetectionBoxes_Recall/AR@100 (small)",
        "DetectionBoxes_Recall/AR@100 (medium)",
        "DetectionBoxes_Recall/AR@100 (large)",
    ],
    title="Validation Recall per Object Size",
    ylabel="Average Recall",
    save_path=GRAPHS_PATH / "eval_recall_size_curves.pdf",
)
display(size_recall_fig)

## 9 · Export

`export_run` reproduces the trained graph (fold + `quantize_backbone` +
`quantize_detection_head`) and writes the best checkpoint to the zoo layout
directly into `qat_per-tensor/`. The defaults pick up `cfg.qat_per_tensor` / `cfg.qat_per_channel`,
so the exported variable structure matches the QAT checkpoint.

In [ ]:
export_result = export_run(run_config, export_dir=QAT_DIR)

print(f"{LABEL} dir: ", export_result.export_dir)
print("SavedModel:", export_result.saved_model_dir)
print("Checkpoint:", export_result.checkpoint)
print("Pipeline:  ", export_result.pipeline_config)

## 10 · Register artifacts and publish

In [ ]:
manifest.add_artifact(f"{LABEL}/pipeline.config", artifact_type="pipeline_config", stage=LABEL)
manifest.add_artifact(f"{LABEL}/graphs", artifact_type="evaluation_plots", stage=LABEL)
manifest.add_artifact(f"{LABEL}/saved_model", artifact_type="tfod_saved_model", stage=LABEL)
manifest.add_artifact(f"{LABEL}/checkpoint", artifact_type="tfod_checkpoint", stage=LABEL)

manifest.save(MANIFEST_PATH)

# Drop the scratch run dir so /kaggle/working == {manifest.<LABEL>.json, <LABEL>/}.
shutil.rmtree(RUN_DIR, ignore_errors=True)

print("Published:")
for p in sorted(WORK.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(WORK))

## 11 · Qualitative evaluation

Predictions on unseen PhenoBench test images, using the exported SavedModel from
`qat/` (fake-quant graph, evaluated in fp32 simulation).

In [ ]:
%matplotlib inline

from agri_vision_edge.tfod.inference import (
    load_saved_model,
    load_label_map,
    detect_image,
)

detect_fn = load_saved_model(str(QAT_DIR / "saved_model"))
category_index = load_label_map(label_map_path)

for image_path in test_imgs[:10]:
    vis, _ = detect_image(
        detect_fn=detect_fn,
        image_path=image_path,
        category_index=category_index,
        image_size=config.image_size,
        score_threshold=0.5,
        max_boxes=60,
    )
    plt.figure(figsize=(16, 16))
    plt.imshow(vis)
    plt.axis("off")
    plt.show()
    plt.close()

## 12 · Discussion

`qat` fake-quantizes the backbone + detection head with the full int8
scheme (per-tensor weights). Compare its
validation mAP against the finetune (`ptq`) and the sibling QAT notebook once
both are merged.

Next step — **INT8 TFLite conversion:** `notebooks/tflite_conversion.py`
reconstructs this `qat/` checkpoint (mirroring the fold + quantize graph
above) and emits the deployable int8 model.